In [264]:
using Healpix
using LinearAlgebra
using StaticArrays
using Base.Threads
using WignerD
using NPZ
using Plots
using Falcons
#using PyPlot
#using PyCall
using BenchmarkTools
using SparseArrays

#hp = pyimport("healpy")
#include("../src/EBC.jl")

In [265]:
mutable struct ConvolutionSky{I<:Int, MC<:Matrix{Complex{Float64}}}
    lmax::I
    alm::MC
    realization::I
end

mutable struct ConvolutionBeam{I<:Int, MC<:Matrix{Complex{Float64}}}
    lmax::I
    mmax::I
    blm::MC
end

#=
mutable struct ConvolutionCalculate{I<:Int, Bl<:Bool}
    nside_output::I
    lmax_calculate::I
    mmax_calculate::I
    HWP::Bl
end
=#

mutable struct ConvolutionCalculate{I<:Int, Bl<:Bool}
    nside_output::I
    lstart::I
    lstop::I
    mmax_calculate::I
    HWP::Bl
end



In [266]:
function ConvolutionSky(;
        lmax::Int = 3*128 - 1,
        alm::Matrix{ComplexF64} = fill(1.0 + 1.0im, 2, 2),
        realization::Int = 1
    )
    return ConvolutionSky(
        lmax,
        alm,
        realization
    )
end

function ConvolutionBeam(;
    lmax::Int = 3*128-1,
    mmax::Int = 2,
    blm::Matrix{ComplexF64} = fill(1.0 + 1.0im, 2, 2)
    )
    return ConvolutionBeam(lmax, mmax, blm)
end

function ConvolutionCalculate(;
    nside_output::Int = 128,
    lstart::Int = 0,
    lstop::Int = 3*nside_output - 1,
    mmax_calculate::Int = 2,
    HWP::Bool = false
)
    lstart <= lstop || throw(ArgumentError("lstart must be <= lstop (got lstart=$lstart, lstop=$lstop)"))
    mmax_calculate <= lstop || throw(ArgumentError("mmax_calculate must be <= lstop (got mmax_calculate=$mmax_calculate, lstop=$lstop)"))
    return ConvolutionCalculate(
        nside_output,
        lstart,
        lstop,
        mmax_calculate,
        HWP
    )
end


ConvolutionCalculate

In [267]:
#methods(ConvolutionSky)

In [268]:
cs = ConvolutionSky()
fieldnames(typeof(cs))

(:lmax, :alm, :realization)

In [269]:
cb = ConvolutionBeam()
fieldnames(typeof(cb))

(:lmax, :mmax, :blm)

In [270]:
cc = ConvolutionCalculate(nside_output = 128, lstart = 0, mmax_calculate = 2)
fieldnames(typeof(cc))

(:nside_output, :lstart, :lstop, :mmax_calculate, :HWP)

In [271]:
include("../../BeamToyModel/src/BeamToyModel.jl")

Main.BeamToyModel

In [272]:
nside_in = 128
lmax_in = 3nside_in-1
fwhm = deg2rad(1.0)

0.017453292519943295

In [273]:
blm_harmonic = BeamToyModel.gaussian_harmonic(lmax_in, fwhm, mmax = lmax_in)

(blmT = Alm{ComplexF64, Vector{ComplexF64}}(ComplexF64[0.28209479177387814 + 0.0im, 0.4885756718694027 + 0.0im, 0.6306791852123736 + 0.0im, 0.7461067059896269 + 0.0im, 0.8458196072043231 + 0.0im, 0.9348319547260208 + 0.0im, 1.0159345688951988 + 0.0im, 1.0908692242925033 + 0.0im, 1.1608087185364095 + 0.0im, 1.226586793157029 + 0.0im  …  0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im], 383, 383, 767), blmE = Alm{ComplexF64, Vector{ComplexF64}}(ComplexF64[0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im  …  0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im], 383, 383, 767), blmB = Alm{ComplexF64, Vector{ComplexF64}}(ComplexF64[0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0i

In [274]:
cb.lmax = lmax_in
cb.mmax = 2
cb.blm = hcat(blm_harmonic.blmT.alm,blm_harmonic.blmE.alm,blm_harmonic.blmB.alm) 
;

In [275]:
nalm = Healpix.numberOfAlms(lmax_in, lmax_in)
almT = rand(ComplexF64, nalm)
almE = rand(ComplexF64, nalm)
almB = rand(ComplexF64, nalm)
cs.lmax = lmax_in
cs.alm = hcat(almT, almE, almB)
;

In [276]:
#=
function lrange_idx(l::Int, lstart::Int)
    l >= lstart || throw(ArgumentError("l must be >= lstart"))
    offset = sum(2k + 1 for k in lstart:l-1; init=0)
    start = offset + 1
    stop  = start + (2l + 1) - 1
    return start, stop
end
=#
#=
function lmr_idx(;l::Int, m::Int, lstart::Int)
    l >= lstart || throw(ArgumentError("l must be >= lstart"))
    (-l <= m <= l) || throw(ArgumentError("m must be in [-l, l]"))
    offset = sum(2k + 1 for k in lstart:l-1; init=0)
    idx = offset + (m + l) + 1   # m=-l -> +1, m=l -> +2l+1
    return idx
end
=#
function lmr_idx(; l::Int, m::Int, lstart::Int, mmax::Int)
    l ≥ lstart || throw(ArgumentError("l must be >= lstart"))
    mcap = min(l, mmax)
    (-mcap ≤ m ≤ mcap) || throw(ArgumentError("m must be in [-min(l,mmax), min(l,mmax)]"))
    offset = 0
    for k in lstart:l-1
        offset += 2*min(k, mmax) + 1
    end
    return offset + (m + mcap) + 1
end
#=
function alm_idx(;l::Integer, m::Integer, lmax::Integer)
    return Int(m * (2 * lmax + 1 - m) // 2 + l)+1
end
=#
function alm_idx(; l::Integer, m::Integer, lmax::Integer, mmax::Integer=lmax)
    (0 ≤ m ≤ mmax) || throw(ArgumentError("m must be in [0, mmax]"))
    (m ≤ l ≤ lmax) || throw(ArgumentError("l must satisfy m ≤ l ≤ lmax"))

    # offset for all previous m-blocks
    offset = m * (lmax + 1) - (m * (m - 1)) ÷ 2
    return Int(offset + (l - m) + 1)  # 1-based
end

function blm_lrange(cb, cc)
    n = sum(2*min(l, cc.mmax_calculate) + 1 for l in cc.lstart:cc.lstop)
    blm_calc = Matrix{ComplexF64}(undef, n, 3)
    for l in cc.lstart:cc.lstop
        m = 0
        idx_in = alm_idx(l=l, m=m, lmax=cb.lmax, mmax = cb.mmax)
        idx_out= lmr_idx(l=l, m=m, lstart=cc.lstart, mmax=cc.mmax_calculate)
        blm_calc[idx_out,1] = cb.blm[idx_in,1]
        blm_calc[idx_out,2] = -(cb.blm[idx_in,2] + 1im*cb.blm[idx_in,3]) # spin2 = -(E +iB)
        blm_calc[idx_out,3] = -(cb.blm[idx_in,2] - 1im*cb.blm[idx_in,3]) # spin-2 = -(E -iB)
        for m in 1:min(l, cc.mmax_calculate)
            phase = isodd(m) ? -1.0 : 1.0
            idx_in = alm_idx(l=l, m=m, lmax=cb.lmax, mmax = cb.mmax)
            idx_out_positive= lmr_idx(l=l, m=m, lstart=cc.lstart, mmax=cc.mmax_calculate)
            idx_out_negative= lmr_idx(l=l, m=-m, lstart=cc.lstart, mmax=cc.mmax_calculate)
            # m positive
            blm_calc[idx_out_positive,1] = cb.blm[idx_in,1]
            blm_calc[idx_out_positive,2] = -(cb.blm[idx_in,2] + 1im*cb.blm[idx_in,3]) # spin2 = -(E +iB)
            blm_calc[idx_out_positive,3] = -(cb.blm[idx_in,2] - 1im*cb.blm[idx_in,3]) # spin-2 = -(E -iB)
            # m negative
            blm_calc[idx_out_negative,1] = conj(cb.blm[idx_in,1])*phase # conj(spin0)*(-1)^m 
            blm_calc[idx_out_negative,2] = conj(blm_calc[idx_out_positive,3])*phase # conj(spin-2)*(-1)^m 
            blm_calc[idx_out_negative,3] = conj(blm_calc[idx_out_positive,2])*phase # conj(spin2)*(-1)^m 
        end
    end
    return blm_calc
end

blm_lrange (generic function with 1 method)

In [277]:
test = blm_lrange(cb, cc)

1914×3 Matrix{ComplexF64}:
 0.282095+0.0im      -0.0-0.0im      -0.0-0.0im
     -0.0+0.0im       0.0-0.0im       0.0-0.0im
 0.488576+0.0im      -0.0-0.0im      -0.0-0.0im
      0.0+0.0im      -0.0-0.0im      -0.0-0.0im
      0.0-0.0im   0.63061+0.0im      -0.0+0.0im
     -0.0+0.0im       0.0-0.0im       0.0-0.0im
 0.630679+0.0im      -0.0-0.0im      -0.0-0.0im
      0.0+0.0im      -0.0-0.0im      -0.0-0.0im
      0.0+0.0im      -0.0-0.0im   0.63061-0.0im
      0.0-0.0im  0.746025+0.0im      -0.0+0.0im
     -0.0+0.0im       0.0-0.0im       0.0-0.0im
 0.746107+0.0im      -0.0-0.0im      -0.0-0.0im
      0.0+0.0im      -0.0-0.0im      -0.0-0.0im
         ⋮                       
      0.0+0.0im      -0.0-0.0im      -0.0-0.0im
      0.0+0.0im      -0.0-0.0im  0.143048-0.0im
      0.0-0.0im  0.140261+0.0im      -0.0+0.0im
     -0.0+0.0im       0.0-0.0im       0.0-0.0im
 0.140276+0.0im      -0.0-0.0im      -0.0-0.0im
      0.0+0.0im      -0.0-0.0im      -0.0-0.0im
      0.0+0.0im      -0.0-0

In [278]:
function alm_lrange(cs, cc)
    n = sum(2*l + 1 for l in cc.lstart:cc.lstop)
    alm_calc = Matrix{ComplexF64}(undef, n, 3)
    for l in cc.lstart:cc.lstop
        m = 0
        idx_in = alm_idx(l=l, m=m, lmax=cs.lmax)
        idx_out= lmr_idx(l=l, m=m, lstart=cc.lstart, mmax=cs.lmax)
        alm_calc[idx_out,1] = cs.alm[idx_in,1]
        alm_calc[idx_out,2] = cs.alm[idx_in,2]
        alm_calc[idx_out,3] = cs.alm[idx_in,3]
        for m in 1:l
            phase = isodd(m) ? -1.0 : 1.0
            idx_in = alm_idx(l=l, m=m, lmax=cs.lmax)
            idx_out_positive= lmr_idx(l=l, m=m, lstart=cc.lstart, mmax=cs.lmax)
            idx_out_negative= lmr_idx(l=l, m=-m, lstart=cc.lstart, mmax=cs.lmax)
            # m positive
            alm_calc[idx_out_positive,1] = cs.alm[idx_in,1]                           # spin0
            alm_calc[idx_out_positive,2] = -(cs.alm[idx_in,2] + 1im*cs.alm[idx_in,3]) # spin2 = -(E +iB)
            alm_calc[idx_out_positive,3] = -(cs.alm[idx_in,2] - 1im*cs.alm[idx_in,3]) # spin2 = -(E +iB)
            # m negative
            alm_calc[idx_out_negative,1] = conj(cs.alm[idx_in,1])*phase             # conj(spin0)
            alm_calc[idx_out_negative,2] = conj(alm_calc[idx_out_positive,3])*phase # conj(spin2)*(-1)^m
            alm_calc[idx_out_negative,3] = conj(alm_calc[idx_out_positive,2])*phase # conj(spin-2)*(-1)^m
        end
    end
    return alm_calc
end

alm_lrange (generic function with 1 method)

In [279]:
alm_lrange(cs, cc)

147456×3 Matrix{ComplexF64}:
   0.222023+0.228283im      0.480291+0.695287im    0.152073+0.645464im
  -0.416595+0.21492im        1.50913-0.176341im   0.0690315-1.04786im
   0.428772+0.0246972im    0.0912916+0.361529im    0.692494+0.191964im
   0.416595+0.21492im     -0.0690315-1.04786im     -1.50913-0.176341im
 0.00448043-0.00490593im    -1.11662+0.201764im   -0.675913+1.59964im
 -0.0514178+0.728572im       1.14623-0.0591124im   -0.11328-1.04512im
   0.872947+0.40833im        0.86497+0.079925im    0.157737+0.294848im
  0.0514178+0.728572im       0.11328-1.04512im     -1.14623-0.0591124im
 0.00448043+0.00490593im   -0.675913-1.59964im     -1.11662-0.201764im
  -0.772726+0.568278im      0.921571+0.81241im    -0.124524-1.11223im
   0.307109-0.422078im     -0.131878-0.153271im    -0.11712+1.41828im
   -0.35025+0.125863im       1.03817-0.573668im    0.410486-1.30975im
   0.843282+0.601561im      0.529388+0.0724536im   0.111449+0.0948204im
           ⋮                                      
 

## wignerD

In [280]:
n_beam = sum(2*min(l, cb.mmax) + 1 for l in cc.lstart:cc.lstop)
D_beam = spzeros(n_beam, n_beam)

n_sky = sum(2*l + 1 for l in cc.lstart:cc.lstop)
D_sky = spzeros(n_sky, n_beam)

147456×1914 SparseMatrixCSC{Float64, Int64} with 0 stored entries:
⎡⠀⠀⎤
⎢⠀⠀⎥
⎢⠀⠀⎥
⎢⠀⠀⎥
⎢⠀⠀⎥
⎢⠀⠀⎥
⎢⠀⠀⎥
⎢⠀⠀⎥
⎢⠀⠀⎥
⎢⠀⠀⎥
⎢⠀⠀⎥
⎢⠀⠀⎥
⎢⠀⠀⎥
⎢⠀⠀⎥
⎢⠀⠀⎥
⎢⠀⠀⎥
⎢⠀⠀⎥
⎢⠀⠀⎥
⎢⠀⠀⎥
⎢⠀⠀⎥
⎢⠀⠀⎥
⎢⠀⠀⎥
⎢⠀⠀⎥
⎢⠀⠀⎥
⎢⠀⠀⎥
⎣⠀⠀⎦

In [281]:
include("../src/function2/rotator.jl")

WignerD_calculator_fast (generic function with 1 method)

In [282]:
ell = 100

100

In [ ]:
θ = pi/3
φ = pi/4

0.7853981633974483

In [ ]:
nptg = 1
θ = pi/3
φ = pi/4
dθ = rand(nptg)*1e-2
dφ = rand(nptg)*1e-2
ψ = rand(nptg)*2pi
;

In [285]:
alphas = zeros(size(dθ))
betas = zeros(size(dθ))
gammas = zeros(size(dθ))

for i in eachindex(dθ)
    err, (alphas[i], betas[i], gammas[i]) = check_split(φ, θ, dφ[i], dθ[i], ψ[i])
    if err >= 1e0
        @show err
    end
end

In [286]:
# first
# 1step version
function test1(ell, φ, θ, ψ)
    D_result = zeros(ComplexF64, 2ell+1, 2ell+1)
    for i in eachindex(φ)
        D_result .+= WignerD.wignerD(ell, φ[i], θ[i], ψ[i])
    end
    return D_result./length(φ)
end

test1 (generic function with 1 method)

In [287]:
D1 = test1(ell, φ.+dφ, θ.+dθ, ψ)

201×201 Matrix{ComplexF64}:
  2.56303e-13-6.40725e-15im  …   4.94117e-16-2.25807e-15im
 -1.52109e-12+1.62755e-12im     -8.30592e-16-1.27114e-15im
 -5.48523e-13-1.28666e-11im     -1.85416e-15-3.71642e-16im
  4.50187e-11+4.06124e-11im      1.99608e-15-1.35504e-15im
 -2.46018e-10+1.48327e-11im     -7.98506e-16+4.38308e-15im
  5.86945e-10-6.74128e-10im  …  -2.10576e-15-2.98736e-15im
  2.29495e-10+2.9427e-9im        5.29753e-15+8.68999e-16im
  -6.89126e-9-5.78985e-9im      -2.26824e-15+1.65942e-15im
    2.5491e-8-2.44035e-9im       6.55193e-16-4.48755e-15im
  -4.31402e-8+5.32224e-8im        1.0517e-15+1.38567e-15im
  -1.95658e-8-1.72327e-7im   …  -3.32017e-15-4.25178e-16im
   3.28928e-7+2.57152e-7im       9.78958e-16-7.70558e-16im
  -9.51606e-7+1.25056e-7im       -2.5149e-16+2.2822e-15im
             ⋮               ⋱              ⋮
 -9.78958e-16-7.70558e-16im      -3.28928e-7+2.57152e-7im
 -3.32017e-15+4.25178e-16im  …   -1.95658e-8+1.72327e-7im
  -1.0517e-15+1.38567e-15im       4.31402e-8

In [288]:
# second
# 2step version
function test2(ell, φ, θ, ψ, φ_pix, θ_pix)
    D_result = zeros(ComplexF64, 2ell+1, 2ell+1)
    #d = WignerD.wignerd(ell, pi/2)
    D_1st = WignerD.wignerD(ell, φ_pix, θ_pix, 0.0)
    for i in eachindex(φ)
        dφ = φ[i] - φ_pix
        dθ = θ[i] - θ_pix
        err, (α, β, γ) = check_split(φ_pix, θ_pix, dφ, dθ, ψ[i])
        D_2nd = WignerD.wignerD(ell, α, β, γ)
        D_result .+=  D_1st * D_2nd
    end
    return D_result./length(φ)
end

test2 (generic function with 1 method)

In [289]:
D2 = test2(ell, φ.+dφ, θ.+dθ, ψ, φ, θ)

201×201 Matrix{ComplexF64}:
   2.5792e-13+2.34528e-16im  …  -1.95164e-15-1.09615e-15im
 -1.52205e-12+1.62727e-12im     -1.61228e-15+5.17279e-16im
 -5.46205e-13-1.28673e-11im      -6.9278e-16+1.76944e-15im
  4.50187e-11+4.06127e-11im     -9.52852e-16-2.20228e-15im
 -2.46018e-10+1.48324e-11im      4.02488e-15+1.90149e-15im
  5.86945e-10-6.74128e-10im  …  -3.43112e-15+1.21198e-15im
  2.29495e-10+2.9427e-9im        2.54715e-15-4.7332e-15im
  -6.89126e-9-5.78985e-9im       7.35542e-16+2.69152e-15im
    2.5491e-8-2.44035e-9im      -3.98754e-15-2.21178e-15im
  -4.31402e-8+5.32224e-8im       1.74903e-15-3.3013e-16im
  -1.95658e-8-1.72327e-7im   …  -1.60345e-15+2.97015e-15im
   3.28928e-7+2.57152e-7im      -1.97737e-16-1.31127e-15im
  -9.51606e-7+1.25056e-7im       2.08199e-15+1.05151e-15im
             ⋮               ⋱              ⋮
  1.97737e-16-1.31127e-15im      -3.28928e-7+2.57152e-7im
 -1.60345e-15-2.97015e-15im  …   -1.95658e-8+1.72327e-7im
 -1.74903e-15-3.3013e-16im        4.31402e-8+

In [290]:
maximum(abs.(D1 .- D2))

1.9189241951615528e-13

In [291]:
function effective_wignerD(ell, φ, θ, ψ, φ_pix, θ_pix)
    D_result = zeros(ComplexF64, 2ell+1, 2ell+1)
    #d = WignerD.wignerd(ell, pi/2)
    D_1st = WignerD.wignerD(ell, φ_pix, θ_pix, 0.0)
    for i in eachindex(φ)
        dφ = φ[i] - φ_pix
        dθ = θ[i] - θ_pix
        err, (α, β, γ) = check_split(φ_pix, θ_pix, dφ, dθ, ψ[i])
        D_2nd = WignerD.wignerD(ell, α, β, γ)
        D_result .+=  D_1st * D_2nd
    end
    return D_result./length(φ)
end

effective_wignerD (generic function with 1 method)

In [292]:
D_effective = effective_wignerD(ell, φ.+dφ, θ.+dθ, ψ, φ, θ);

In [293]:
n_beam = sum(2*min(l, cb.mmax) + 1 for l in cc.lstart:cc.lstop)
D_beam = spzeros(n_beam, n_beam)

1914×1914 SparseMatrixCSC{Float64, Int64} with 0 stored entries:
⎡⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎦

In [294]:
function effective_wignerD_dir(ell, φ, θ, ψ, φ_pix, θ_pix)
    D_result = zeros(ComplexF64, 2ell+1, 2ell+1)
    #d = WignerD.wignerd(ell, pi/2)
    D_1st = WignerD.wignerD(ell, φ_pix, θ_pix, 0.0)
    for i in eachindex(φ)
        dφ = φ[i] - φ_pix
        dθ = θ[i] - θ_pix
        err, (α, β, γ) = check_split(φ_pix, θ_pix, dφ, dθ, ψ[i])
        D_2nd = WignerD.wignerD(ell, α, β, γ)
        D_result .+=  D_1st * D_2nd
    end
    return D_result./length(φ)
end

effective_wignerD_dir (generic function with 1 method)

In [295]:
d = WignerD.wignerd(ell, pi/2);

In [296]:
Dp = @time WignerD_calculator_fast(d, ell, φ, θ, ψ[1]);

  0.056155 seconds (23.73 k allocations: 2.904 MiB, 87.78% compilation time)


In [297]:
ellWignerD_calculator!

LoadError: UndefVarError: `ellWignerD_calculator!` not defined

In [ ]:
# 複数角度サンプルの単純平均:
#   D_eff = (1/N) * Σ_i D(phi[i], theta[i], psi[i])
function WignerD_calculator!(result::AbstractMatrix{ComplexF64},
                             d, ell::Int,
                             phi::AbstractVector, theta::AbstractVector, psi::AbstractVector;
                             tmpD=nothing,
                             tmpB=nothing, w=nothing, p=nothing, q=nothing)
    N = length(phi)
    @assert length(theta) == N && length(psi) == N

    L = 2*ell + 1
    @assert size(result,1) == L && size(result,2) == L
    tmpD === nothing && (tmpD = Matrix{ComplexF64}(undef, L, L))

    fill!(result, 0.0 + 0.0im)
    @inbounds for i in eachindex(phi)
        WignerD_calculator!(tmpD, d, ell, phi[i], theta[i], psi[i];
                            tmpB=tmpB, w=w, p=p, q=q)
        result .+= tmpD
    end
    result ./= N
    return result
end

# 2step 平均（effective_wignerD 相当）:
#   D_eff = (1/N) * Σ_i D(phi_pix, theta_pix, 0) * D(α_i, β_i, γ_i)
#   where (α_i, β_i, γ_i) = check_split(phi_pix, theta_pix, dphi_i, dtheta_i, psi_i)
function WignerD_calculator!(result::AbstractMatrix{ComplexF64},
                             d, ell::Int,
                             phi::AbstractVector, theta::AbstractVector, psi::AbstractVector,
                             phi_pix, theta_pix;
                             eps=1e-12,
                             D1=nothing, D2=nothing, tmpMul=nothing,
                             tmpB1=nothing, w1=nothing, p1=nothing, q1=nothing,
                             tmpB2=nothing, w2=nothing, p2=nothing, q2=nothing)
    N = length(phi)
    @assert length(theta) == N && length(psi) == N

    L = 2*ell + 1
    @assert size(result,1) == L && size(result,2) == L
    D1     === nothing && (D1     = Matrix{ComplexF64}(undef, L, L))
    D2     === nothing && (D2     = Matrix{ComplexF64}(undef, L, L))
    tmpMul === nothing && (tmpMul = Matrix{ComplexF64}(undef, L, L))

    # 1st step: pixel center fixed rotation
    WignerD_calculator!(D1, d, ell, phi_pix, theta_pix, 0.0;
                        tmpB=tmpB1, w=w1, p=p1, q=q1)

    fill!(result, 0.0 + 0.0im)
    @inbounds for i in eachindex(phi)
        dphi = phi[i] - phi_pix
        dtheta = theta[i] - theta_pix
        _, (α, β, γ) = check_split(phi_pix, theta_pix, dphi, dtheta, psi[i]; eps=eps)

        WignerD_calculator!(D2, d, ell, α, β, γ;
                            tmpB=tmpB2, w=w2, p=p2, q=q2)
        mul!(tmpMul, D1, D2)
        result .+= tmpMul
    end

    result ./= N
    return result
end

WignerD_calculator! (generic function with 3 methods)

In [ ]:
L = 2*ell + 1
result = zeros(ComplexF64, L, L)

# 単純平均
A = WignerD_calculator!(result, d, ell, φ.+dφ, θ.+dθ, ψ)

result = zeros(ComplexF64, L, L)
# 2step平均
B = WignerD_calculator!(result, d, ell, φ.+dφ, θ.+dθ, ψ, φ, θ)

201×201 Matrix{ComplexF64}:
  1.49142e-13-2.77149e-13im  …  -1.03656e-16-1.62421e-15im
  8.28786e-13+2.40673e-12im     -7.86125e-16+5.38346e-16im
 -1.32363e-11-6.30634e-12im      5.96329e-18-1.19001e-15im
  6.46259e-11-2.35783e-11im      2.32245e-16+2.32219e-15im
 -1.15437e-10+2.53888e-10im     -3.25253e-15-1.64224e-15im
 -3.62436e-10-9.41391e-10im  …   2.61746e-15-1.31743e-15im
   3.04809e-9+1.32135e-9im      -1.50468e-15+4.12591e-15im
  -9.36316e-9+3.79624e-9im      -2.33477e-15-1.40929e-15im
   1.09355e-8-2.65006e-8im       3.07861e-15+1.082e-15im
   2.99905e-8+7.03697e-8im      -6.55896e-16+2.3713e-15im
  -1.79798e-7-7.04999e-8im   …   6.73222e-16-1.66946e-15im
   4.23275e-7-1.89306e-7im       1.70134e-15+6.19086e-16im
  -3.70503e-7+9.96397e-7im      -7.85557e-16-3.55767e-16im
             ⋮               ⋱              ⋮
 -1.70206e-15+6.19155e-16im      -4.23275e-7-1.89306e-7im
  6.74238e-16+1.67025e-15im  …   -1.79798e-7+7.04999e-8im
  6.53933e-16+2.37136e-15im      -2.99905e-8+7

In [ ]:
maximum(abs.(A.-B))

1.0927797337298396e-13

In [ ]:
function effective_wignerD(ell, φ, θ, ψ, φ_pix, θ_pix)
    D_result = zeros(ComplexF64, 2ell+1, 2ell+1)
    #d = WignerD.wignerd(ell, pi/2)
    D_1st = WignerD.wignerD(ell, φ_pix, θ_pix, 0.0)
    for i in eachindex(φ)
        dφ = φ[i] - φ_pix
        dθ = θ[i] - θ_pix
        err, (α, β, γ) = check_split(φ_pix, θ_pix, dφ, dθ, ψ[i])
        D_2nd = WignerD.wignerD(ell, α, β, γ)
        D_result .+=  D_1st * D_2nd
    end
    return D_result./length(φ)
end

In [325]:
d_beam = spzeros(ComplexF64,n_beam, n_beam)


1914×1914 SparseMatrixCSC{ComplexF64, Int64} with 0 stored entries:
⎡⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎦

In [323]:
D_beam = spzeros(ComplexF64,n_beam, n_beam)

1914×1914 SparseMatrixCSC{ComplexF64, Int64} with 0 stored entries:
⎡⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎦

In [ ]:
function D_2ndtest(cb, cc, α, β, γ)
    D_beam = spzeros(ComplexF64,n_beam, n_beam)
    for l in cc.lstart:cc.lstop
        @show l
        #d_temp = WignerD.wignerd(ell,pi/2)
        m_ = min(l, cb.mmax)
        @inbounds for m in -m_:m_
            m_idx = lmr_idx(l=l, m=m, lstart=cc.lstart, mmax=cb.mmax)
            @show m_idx
            @inbounds for n in -m_:m_
                n_idx = lmr_idx(l=l, m=n, lstart=cc.lstart, mmax=cb.mmax)
                D_beam[m_idx, n_idx] = WignerD.wignerDjmn(l, m, n, α, β, γ)
            end
        end
    end
    return D_beam
end

l = 0
m_idx = 1
l = 1
m_idx = 2
m_idx = 3
m_idx = 4
l = 2
m_idx = 5
m_idx = 6
m_idx = 7
m_idx = 8
m_idx = 9
l = 3
m_idx = 10
m_idx = 11
m_idx = 12
m_idx = 13
m_idx = 14
l = 4
m_idx = 15
m_idx = 16
m_idx = 17
m_idx = 18
m_idx = 19
l = 5
m_idx = 20
m_idx = 21
m_idx = 22
m_idx = 23
m_idx = 24
l = 6
m_idx = 25
m_idx = 26
m_idx = 27
m_idx = 28
m_idx = 29
l = 7
m_idx = 30
m_idx = 31
m_idx = 32
m_idx = 33
m_idx = 34
l = 8
m_idx = 35
m_idx = 36
m_idx = 37
m_idx = 38
m_idx = 39
l = 9
m_idx = 40
m_idx = 41
m_idx = 42
m_idx = 43
m_idx = 44
l = 10
m_idx = 45
m_idx = 46
m_idx = 47
m_idx = 48
m_idx = 49
l = 11
m_idx = 50
m_idx = 51
m_idx = 52
m_idx = 53
m_idx = 54
l = 12
m_idx = 55
m_idx = 56
m_idx = 57
m_idx = 58
m_idx = 59
l = 13
m_idx = 60
m_idx = 61
m_idx = 62
m_idx = 63
m_idx = 64
l = 14
m_idx = 65
m_idx = 66
m_idx = 67
m_idx = 68
m_idx = 69
l = 15
m_idx = 70
m_idx = 71
m_idx = 72
m_idx = 73
m_idx = 74
l = 16
m_idx = 75
m_idx = 76
m_idx = 77
m_idx = 78
m_idx = 79
l = 17
m_idx = 80
m_idx = 81
m_

In [330]:
function D_1sttest(cb, cc, φ, θ, ψ)
    D_beam = spzeros(ComplexF64,n_beam, n_beam)
    for l in cc.lstart:cc.lstop
        @show l
        #d_temp = WignerD.wignerd(ell,pi/2)
        m_ = min(l, cb.mmax)
        @inbounds for m in -m_:m_
            m_idx = lmr_idx(l=l, m=m, lstart=cc.lstart, mmax=cb.mmax)
            @show m_idx
            @inbounds for n in -m_:m_
                n_idx = lmr_idx(l=l, m=n, lstart=cc.lstart, mmax=cb.mmax)
                D_beam[m_idx, n_idx] = WignerD.wignerDjmn(l, m, n, φ, θ, ψ)
            end
        end
    end
    return D_beam
end

D_1sttest (generic function with 1 method)

In [338]:
nptg = 1
θ = pi/3
φ = pi/4
dθ = rand()*1e-2
dφ = rand()*1e-2
ψ = rand()*2pi

0.23101319433527118

In [340]:
err_, (alphas_, betas_, gammas_) = check_split(φ, θ, dφ, dθ, ψ)

(6.2727600891321345e-15, (0.7103217222390543, 0.00835981081916969, -0.47618992006922145))

In [327]:
d_beam == D_beam

true

In [314]:
WignerD.wignerDjmn(4, 1, 1, pi/3, pi/2, pi/6)

2.2962127484012914e-17 - 0.37500000000000067im

In [311]:
fieldnames(typeof(WignerD.wignerdjmn))

()

1914×1914 SparseMatrixCSC{Float64, Int64} with 0 stored entries:
⎡⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎦

In [159]:
fieldnames(typeof(cb))

(:lmax, :mmax, :blm)

In [157]:
cc.mmax_calculate

2

In [134]:
D_beam[1,1:4] .=+ Vector(1:1:4).*0.0

4-element view(::SparseMatrixCSC{Float64, Int64}, 1, 1:4) with eltype Float64:
 0.0
 0.0
 0.0
 0.0

In [135]:
D_beam[1,1:4]

4-element SparseVector{Float64, Int64} with 0 stored entries

In [153]:
lmr_idx(l=2, m=0, lstart=0, mmax=2)

7

In [58]:
WignerD_calculator!(result::AbstractMatrix{ComplexF64},
                             d, ell::Int, phi, theta, psi;
                             tmpB=nothing, w=nothing, p=nothing, q=nothing)

LoadError: UndefVarError: `result` not defined

In [33]:
cc.lstop

383

In [23]:
D

1000×1000 SparseMatrixCSC{Float64, Int64} with 0 stored entries:
⎡⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎦

In [52]:
function lmr_idx(;l::Int, m::Int, lstart::Int)
    l >= lstart || throw(ArgumentError("l must be >= lstart"))
    (-l <= m <= l) || throw(ArgumentError("m must be in [-l, l]"))
    offset = sum(2k + 1 for k in lstart:l-1; init=0)
    idx = offset + (m + l) + 1   # m=-l -> +1, m=l -> +2l+1
    return idx
end

lmr_idx (generic function with 1 method)

In [53]:
lmr_idx(l=2, m=0, lstart=0)

7

In [55]:
lmr_idx

lmr_idx (generic function with 1 method)

In [30]:
lmr_idx(l=2, m=0, lstart=cc.lstart, mmax=cc.mmax_calculate)

7

In [37]:
lmr_idx(l=3, m=2, lstart=cc.lstart, mmax=cc.mmax_calculate)

14

In [32]:
lmr_idx(l=2, m=-2, lstart=cc.lstart, mmax=cc.mmax_calculate)

5

In [33]:
lmr_idx(l=2, m=-1, lstart=cc.lstart, mmax=cc.mmax_calculate)

6

In [25]:
n = sum(2*min(l, 3) + 1 for l in cc.lstart:cc.lstop)

2676

In [26]:
1+3+5+7+7*(cc.lstop-3)

2676

In [ ]:
n = sum(2*min(l, cb.mmax) + 1 for l in cc.lstart:cc.lstop)


147456

In [99]:
alm_idx(l=4, m=3, lmax=4, mmax=3)

14

In [70]:
m = -2
phase = isodd(m) ? -1.0 : 1.0


1.0

In [ ]:
idx_start, idx_stop = lrange_idx(3, 2)

(6, 12)

In [58]:
lmr_idx(2, -2, 0)

5

In [46]:
maximum(cc.l_calculate)

383

In [ ]:
function lrange_idx(l::Int, lstart::Int)
    l >= lstart || throw(ArgumentError("l must be >= lstart"))
    offset = sum(2k + 1 for k in lstart:l-1)
    start = offset + 1
    stop  = start + (2l + 1) - 1
    return start, stop
end


In [43]:
for l in cc.l_calculate
    println("l = $l")
end

l = 0
l = 1
l = 2
l = 3
l = 4
l = 5
l = 6
l = 7
l = 8
l = 9
l = 10
l = 11
l = 12
l = 13
l = 14
l = 15
l = 16
l = 17
l = 18
l = 19
l = 20
l = 21
l = 22
l = 23
l = 24
l = 25
l = 26
l = 27
l = 28
l = 29
l = 30
l = 31
l = 32
l = 33
l = 34
l = 35
l = 36
l = 37
l = 38
l = 39
l = 40
l = 41
l = 42
l = 43
l = 44
l = 45
l = 46
l = 47
l = 48
l = 49
l = 50
l = 51
l = 52
l = 53
l = 54
l = 55
l = 56
l = 57
l = 58
l = 59
l = 60
l = 61
l = 62
l = 63
l = 64
l = 65
l = 66
l = 67
l = 68
l = 69
l = 70
l = 71
l = 72
l = 73
l = 74
l = 75
l = 76
l = 77
l = 78
l = 79
l = 80
l = 81
l = 82
l = 83
l = 84
l = 85
l = 86
l = 87
l = 88
l = 89
l = 90
l = 91
l = 92
l = 93
l = 94
l = 95
l = 96
l = 97
l = 98
l = 99
l = 100
l = 101
l = 102
l = 103
l = 104
l = 105
l = 106
l = 107
l = 108
l = 109
l = 110
l = 111
l = 112
l = 113
l = 114
l = 115
l = 116
l = 117
l = 118
l = 119
l = 120
l = 121
l = 122
l = 123
l = 124
l = 125
l = 126
l = 127
l = 128
l = 129
l = 130
l = 131
l = 132
l = 133
l = 134
l = 135
l = 136
l = 137
l = 13

In [62]:
hcat(blm_harmonic.blmT.alm,
     blm_harmonic.blmE.alm,
     blm_harmonic.blmB.alm) 

73920×3 Matrix{ComplexF64}:
 0.282095+0.0im  0.0+0.0im  0.0+0.0im
 0.488576+0.0im  0.0+0.0im  0.0+0.0im
 0.630679+0.0im  0.0+0.0im  0.0+0.0im
 0.746107+0.0im  0.0+0.0im  0.0+0.0im
  0.84582+0.0im  0.0+0.0im  0.0+0.0im
 0.934832+0.0im  0.0+0.0im  0.0+0.0im
  1.01593+0.0im  0.0+0.0im  0.0+0.0im
  1.09087+0.0im  0.0+0.0im  0.0+0.0im
  1.16081+0.0im  0.0+0.0im  0.0+0.0im
  1.22659+0.0im  0.0+0.0im  0.0+0.0im
  1.28882+0.0im  0.0+0.0im  0.0+0.0im
  1.34798+0.0im  0.0+0.0im  0.0+0.0im
  1.40444+0.0im  0.0+0.0im  0.0+0.0im
         ⋮                  
      0.0+0.0im  0.0+0.0im  0.0+0.0im
      0.0+0.0im  0.0+0.0im  0.0+0.0im
      0.0+0.0im  0.0+0.0im  0.0+0.0im
      0.0+0.0im  0.0+0.0im  0.0+0.0im
      0.0+0.0im  0.0+0.0im  0.0+0.0im
      0.0+0.0im  0.0+0.0im  0.0+0.0im
      0.0+0.0im  0.0+0.0im  0.0+0.0im
      0.0+0.0im  0.0+0.0im  0.0+0.0im
      0.0+0.0im  0.0+0.0im  0.0+0.0im
      0.0+0.0im  0.0+0.0im  0.0+0.0im
      0.0+0.0im  0.0+0.0im  0.0+0.0im
      0.0+0.0im  0.0+0.0im  0.0

In [59]:
[blm_harmonic.blmT.alm, blm_harmonic.blmE.alm, blm_harmonic.blmB.alm]

3-element Vector{Vector{ComplexF64}}:
 [0.28209479177387814 + 0.0im, 0.4885756718694027 + 0.0im, 0.6306791852123736 + 0.0im, 0.7461067059896269 + 0.0im, 0.8458196072043231 + 0.0im, 0.9348319547260208 + 0.0im, 1.0159345688951988 + 0.0im, 1.0908692242925033 + 0.0im, 1.1608087185364095 + 0.0im, 1.226586793157029 + 0.0im  …  0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im]
 [0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im  …  0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im]
 [0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im  …  0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im]

In [35]:
blm_harmonic.blmT.alm

73920-element Vector{ComplexF64}:
 0.28209479177387814 + 0.0im
  0.4885756718694027 + 0.0im
  0.6306791852123736 + 0.0im
  0.7461067059896269 + 0.0im
  0.8458196072043231 + 0.0im
  0.9348319547260208 + 0.0im
  1.0159345688951988 + 0.0im
  1.0908692242925033 + 0.0im
  1.1608087185364095 + 0.0im
   1.226586793157029 + 0.0im
   1.288820860640887 + 0.0im
  1.3479829400254342 + 0.0im
  1.4044432431790195 + 0.0im
                     ⋮
                 0.0 + 0.0im
                 0.0 + 0.0im
                 0.0 + 0.0im
                 0.0 + 0.0im
                 0.0 + 0.0im
                 0.0 + 0.0im
                 0.0 + 0.0im
                 0.0 + 0.0im
                 0.0 + 0.0im
                 0.0 + 0.0im
                 0.0 + 0.0im
                 0.0 + 0.0im

In [47]:
mmax = lmax
nalm = Healpix.numberOfAlms(lmax, mmax)
blmT = Healpix.Alm(lmax, mmax, zeros(ComplexF64, nalm))

Alm{ComplexF64, Vector{ComplexF64}}(ComplexF64[0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im  …  0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im], 383, 383, 767)

In [52]:
blmT

Alm{ComplexF64, Vector{ComplexF64}}(ComplexF64[0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im  …  0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im], 383, 383, 767)

In [53]:
typeof(blmT)

Alm{ComplexF64, Vector{ComplexF64}}

70000*70000*8

In [39]:
70000*70000

4900000000

In [4]:
function ConvolutionSky(;
    nside::Int = 128,
    lmax::Int = 3*nside - 1,
    alm::Matrix{ComplexF64} = fill(1.0 + 1.0im, 2, 2),
    realization::Int = 1
)
    return ConvolutionSky(nside, lmax, alm, realization)
end

ConvolutionSky

In [25]:
mutable struct ConvolutionSky{I<:Int, MC<:Matrix{Complex{Float64}}}
    nside::I
    lmax::I
    alm::MC
    realization::I
end

LoadError: invalid redefinition of constant Main.ConvolutionSky

In [40]:
nalm = Healpix.numberOfAlms(1000, 1000)


501501

In [5]:
nside = 16
cp = gen_ConvolutionParams_pc(nside = nside);
cp.beam_mmax = 10

10

In [6]:
lmax = nside*3-1
res = Resolution(nside)
beammmax = 10

10

In [7]:
# input beam
size = alm_idx(lmax,lmax,lmax)
blm_ = zeros(ComplexF64, 3, size)
blm = hp.blm_gauss(deg2rad(0.5), lmax = lmax, pol = true)
blm_[1,1:length(blm[1,:])] = blm[1,:]
blm_[2,1:length(blm[1,:])] = -blm[2,:]*sqrt(2)
blm_[3,1:length(blm[1,:])] = -blm[3,:]*sqrt(2);
@time blm_full = get_reorder_blm_optimized(blm_, lmax, beammmax);

LoadError: UndefVarError: `hp` not defined

In [8]:
alm = npzread("./inputs/alm=CMB=r0=nside$nside.npy")
@time alm_full = get_reorder_alm_optimized(alm, lmax);

  0.233391 seconds (245.43 k allocations: 16.078 MiB, 3.35% gc time, 98.94% compilation time)


In [9]:
data_path = "/data/n/n339/wigner_file/"

"/data/n/n339/wigner_file/"

In [10]:
@time initial_wignerd = [zeros(ComplexF64,2*i+1,2*i+1) for i in 0:cp.lmax]
@time initial_wignerd_ = [zeros(ComplexF64,2*i+1,2*i+1) for i in 0:cp.lmax]
@time eff_wignerD = [zeros(ComplexF64,2*i+1,2*i+1) for i in 0:cp.lmax];
@time eff_wignerD_ = [zeros(ComplexF64,2*i+1,2*i+1) for i in 0:cp.lmax];

  0.071945 seconds (29.82 k allocations: 4.259 MiB, 40.88% gc time, 57.19% compilation time)
  0.029705 seconds (21.82 k allocations: 3.724 MiB, 97.19% compilation time)
  0.028619 seconds (21.82 k allocations: 3.724 MiB, 96.99% compilation time)
  0.036557 seconds (21.82 k allocations: 3.724 MiB, 96.14% compilation time)


In [11]:
initial_wignerd[2]

3×3 Matrix{ComplexF64}:
 0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im  0.0+0.0im  0.0+0.0im

In [12]:
function initialwignerds_array_(cp, theta, path, initial_wignerd)
    count = 1
    for l in cp.l_range[1]:cp.l_range[2]
        initial_wignerd[count] = swignerd_calc(l, theta, path)
        count += 1
    end
   return initial_wignerd
end

initialwignerds_array_ (generic function with 1 method)

In [13]:
function effective_wignerD_onz_(cp, ψs, initial_wignerd_onz_)
    for m in -cp.lmax-2:cp.lmax+2
        initial_wignerd_onz_[cp.lmax+1+m] = mean(exp.(-1im*ψs*m))
    end
    return initial_wignerd_onz_
end

effective_wignerD_onz_ (generic function with 1 method)

In [14]:
npix = nside2npix(nside)
utv = unique_theta_val(cp);

In [15]:
initial_wignerd_onz = zeros(ComplexF64, 3, 2*cp.lmax+1)
conv_binned_map = zeros(3,npix)
result_d = zeros(ComplexF64,3)
@time for i in 1:length(utv)
    @show i
    start_pix, stop_pix = unique_theta_num(i, cp)
    initialwignerds_array(cp, utv[i], data_path, initial_wignerd);
    for pix_num in start_pix:stop_pix
        center_th, center_phi = pix2angRing(res, pix_num)
        calc_psi = rand(100)*2pi
        effective_wignerD_onz(cp, calc_psi[:], initial_wignerd_onz)
        for j in 1:3
            get_pc_total_effective_wignerD(cp, center_phi, initial_wignerd_onz[j,:], initial_wignerd, eff_wignerD);
            result_d[j] = eff_convolver_optimized(alm_full, blm_full, eff_wignerD, beammmax)
        end
        conv = binned_mapmaker(calc_psi, result_d)
        conv_binned_map[:,pix_num] .= conv
    end
end

i = 1


LoadError: SystemError: opening file "/data/n/n339/wigner_file/dmatrices=ell0.npy": No such file or directory

In [ ]:
alm_smooth = hp.smoothalm([alm[1,:],alm[2,:],alm[3,:]],fwhm = deg2rad(0.5))
in_map = hp.alm2map([alm_smooth[1],alm_smooth[2],alm_smooth[3]], nside = nside)

LoadError: UndefVarError: `alm` not defined